# Stratified CUPED — Experiment Analysis & Monte Carlo Evaluation

This notebook has **two parts**:

| Part | Purpose |
|------|---------|
| **Part 1 — Single-experiment inference** | Analyze one experiment: τ̂, SE, CI, p-value for existing users, new users, and overall |
| **Part 2 — Monte Carlo evaluation** | Run 1000 simulations to verify coverage, SE calibration, and RMSE across estimators |

**Edit the configuration in Section 1, then Run All.**

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "font.size": 9,
})

from config import SimulationConfig
from simulation import simulate_user_level_data
from estimators import (
    diff_in_means, cuped_estimator, cuped_stratified, bayesian_estimator,
    EstimatorResult, StratifiedResult, _two_sided_p,
)
from evaluation import run_simulations
from inference import experiment_report, inference_table, analyze
from plots import plot_estimator_distributions, plot_summary_bars, plot_se_calibration

print("All imports OK.")

## 1. Configuration

Edit any parameter and re-run.

| Parameter | Controls |
|-----------|----------|
| `n_users` | Sample size per experiment |
| `tau_true` | True PATE |
| `beta_pre` | Slope of Y_pre in DGP → autocorrelation |
| `sigma_post` | Noise SD |
| `sigma_heterogeneity` | Heterogeneous effects (0 = homogeneous) |
| `p_new_users` | Fraction without Y_pre |
| `prior_sd_tau` | Bayesian prior SD |

In [ ]:
cfg = SimulationConfig(
    n_users       = 10_000,
    p_treatment   = 0.5,
    tau_true      = 0.05,
    mu_pre        = 1.0,
    sigma_pre     = 0.5,
    beta_pre      = 0.8,
    sigma_post    = 1.0,
    sigma_heterogeneity = 0.0,
    p_new_users   = 0.20,
    random_seed   = 42,
    prior_mean_tau = 0.0,
    prior_sd_tau   = 0.10,
)

N_SIM = 1000

print(f"DGP: Y = {cfg.beta_pre}·Y_pre + τ_i·D + ε,  ε ~ N(0, {cfg.sigma_post}²)")
print(f"     τ_i = {cfg.tau_true} + {cfg.sigma_heterogeneity}·z_i")
print(f"     ρ(Y_pre, Y) ≈ {cfg.effective_autocorrelation():.4f}")
print(f"     Pr(new user) = {cfg.p_new_users},  n = {cfg.n_users}")

---
# Part 1 — Single-Experiment Inference

Simulate one experiment and produce the **experiment report**: treatment effect
on existing users, new users, and overall — each with τ̂, SE, 95% CI, t-stat, p-value.

In [ ]:
rng = np.random.default_rng(cfg.random_seed)
data = simulate_user_level_data(cfg, rng)

n = len(data)
n_new = int(data["is_new_user"].sum())
n_ret = n - n_new
n_treat = int(data["D"].sum())

print(f"Dataset: {n:,} users")
print(f"  Treated: {n_treat:,}  |  Control: {n - n_treat:,}")
print(f"  Returning: {n_ret:,} ({100*n_ret/n:.1f}%)  |  New: {n_new:,} ({100*n_new/n:.1f}%)")
data.head(8)

### 1.1 Experiment Report

In [ ]:
report = experiment_report(data, cfg)

report.style.format({
    "τ̂": "{:.6f}", "SE": "{:.6f}",
    "95% CI lower": "{:.6f}", "95% CI upper": "{:.6f}",
    "t-stat": "{:.3f}", "p-value": "{:.4f}", "n": "{:.0f}",
}).set_caption(f"Experiment Report   (DGP true τ = {cfg.tau_true})")

### 1.2 How to read the report

| Row | What it answers | Method | SE formula |
|-----|-----------------|--------|------------|
| **Existing users** | Is the treatment effective among returning users? | CUPED: $Y = \alpha + \beta D + \mu Y_{\text{pre}}$ → $\hat\beta$ | OLS: $\sqrt{\hat\sigma_e^2 [(X'X)^{-1}]_{22}}$ |
| **New users** | Is the treatment effective among new users? | Diff-in-means | Welch: $\sqrt{s_T^2/n_T + s_C^2/n_C}$ |
| **Overall ATE (pop.)** | What is the overall PATE? | $w_R \hat\beta_R + w_N \hat\tau_N$ | $\sqrt{w_R^2 \text{SE}_R^2 + w_N^2 \text{SE}_N^2}$ |
| **Overall ATE (prec.)** | Min-variance combination | Inverse-variance weighted | $\sqrt{1/(1/V_R + 1/V_N)}$ |
| **Naive** | Benchmark — ignores Y_pre | Diff-in-means on all users | Welch |
| **Bayesian** | Posterior with prior $\tau \sim N(m_0, v_0)$ | Conjugate normal | Posterior SD |

**Signif. codes:** `***` p < 0.001, `**` p < 0.01, `*` p < 0.05, `.` p < 0.10

### 1.3 Confidence interval forest plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.55 * len(report) + 1.2))

labels = report.index.tolist()[::-1]
taus   = report["τ̂"].values[::-1]
lowers = report["95% CI lower"].values[::-1]
uppers = report["95% CI upper"].values[::-1]
pvals  = report["p-value"].values[::-1]
y_pos  = np.arange(len(labels))

colors = ["#c0392b" if p < 0.05 else "#7f8c8d" for p in pvals]

for i, (tau, lo, hi, col) in enumerate(zip(taus, lowers, uppers, colors)):
    ax.plot([lo, hi], [i, i], color=col, linewidth=2.5, solid_capstyle="round")
    ax.plot(tau, i, "o", color=col, markersize=7, zorder=5)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.axvline(cfg.tau_true, color="blue", linewidth=0.8, linestyle=":", alpha=0.7,
           label=f"True τ = {cfg.tau_true}")
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Treatment effect (τ̂) with 95% CI")
ax.set_title("Experiment Report — Forest Plot", fontsize=11)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 1.4 Test: are treatment effects equal for existing vs. new users?

$H_0\!: \beta_R = \tau_N$ (same effect across strata).
Under independence of strata: $t = (\hat\beta_R - \hat\tau_N) / \sqrt{\text{SE}_R^2 + \text{SE}_N^2}$.

In [ ]:
if data["Y_pre"].isna().any():
    strat = cuped_stratified(data)
    diff = strat.returning.tau_hat - strat.new.tau_hat
    se_diff = np.sqrt(strat.returning.se**2 + strat.new.se**2)
    t_diff = diff / se_diff
    p_diff = _two_sided_p(t_diff)
    ci_lo = diff - 1.96 * se_diff
    ci_hi = diff + 1.96 * se_diff

    print("Test of equal treatment effects: H₀: β_R = τ_N")
    print(f"  Difference : {diff:.6f}")
    print(f"  SE(diff)   : {se_diff:.6f}")
    print(f"  95% CI     : [{ci_lo:.6f}, {ci_hi:.6f}]")
    print(f"  t-stat     : {t_diff:.3f}")
    print(f"  p-value    : {p_diff:.4f}")
    print(f"  Conclusion : {'Reject H₀ — effects differ' if p_diff < 0.05 else 'Fail to reject H₀ — no evidence of difference'}")
else:
    print("No new users in this dataset — test not applicable.")

### 1.5 Using with your own data

To analyze a real experiment, load your data into a DataFrame with columns
`D` (treatment 0/1), `Y` (outcome), `Y_pre` (pre-period outcome, NaN for new users):

```python
import pandas as pd
from inference import experiment_report
from config import SimulationConfig

data = pd.read_csv("my_experiment.csv")  # must have D, Y, Y_pre
cfg = SimulationConfig(prior_sd_tau=0.10) # only need Bayesian prior params
report = experiment_report(data, cfg)
print(report)
```

---
# Part 2 — Monte Carlo Evaluation

Run `N_SIM` simulations to assess the **operating characteristics** of each estimator:
bias, RMSE, SE calibration (SE/SD ratio), coverage, and power.

In [ ]:
%%time
df, summary = run_simulations(cfg, n_sim=N_SIM)
print(f"Completed {N_SIM} simulations.")

### 2.1 Summary Table

- **se_sd_ratio ≈ 1**: analytic SE matches empirical variability
- **coverage ≈ 0.95**: the 95% CI has correct coverage
- **power**: fraction of experiments that detect the effect

In [ ]:
summary.style.format("{:.5f}").background_gradient(
    subset=["rmse"], cmap="YlOrRd_r"
).background_gradient(
    subset=["coverage"], cmap="RdYlGn", vmin=0.90, vmax=1.0
).background_gradient(
    subset=["se_sd_ratio"], cmap="RdYlGn", vmin=0.85, vmax=1.15
)

### 2.2 Sampling Distributions

In [ ]:
fig = plot_estimator_distributions(df, cfg)
plt.show()

### 2.3 Summary Bar Charts (Bias, RMSE, Coverage, SE/SD)

In [ ]:
fig = plot_summary_bars(summary)
plt.show()

### 2.4 SE Calibration Scatter

Analytic SE vs. $|\hat\tau - \tau|$. If well-calibrated, ~68% of points should fall below the diagonal.

In [ ]:
fig = plot_se_calibration(df, cfg)
plt.show()

---
# Part 3 — Sensitivity Analysis

### 3.1 Sensitivity to `p_new_users`

In [ ]:
param_name = "p_new_users"
param_values = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
n_sim_sweep = 500

sweep_results = []
for val in param_values:
    cfg_i = SimulationConfig(**{**cfg.__dict__, param_name: val})
    _, summ_i = run_simulations(cfg_i, n_sim=n_sim_sweep)
    for est_name, row in summ_i.iterrows():
        sweep_results.append({param_name: val, "estimator": est_name, **row.to_dict()})

sweep_df = pd.DataFrame(sweep_results)
print(f"Sweep complete: {len(param_values)} values × {n_sim_sweep} sims.")

In [ ]:
focus = ["naive", "cuped_population", "cuped_precision", "bayes"]
metrics = ["rmse", "coverage", "se_sd_ratio", "power"]
titles = ["RMSE", "Coverage (95% CI)", "SE / SD ratio", "Power"]
refs = [None, 0.95, 1.0, None]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, m, t, r in zip(axes.ravel(), metrics, titles, refs):
    for est in focus:
        sub = sweep_df[sweep_df["estimator"] == est]
        if not sub.empty:
            ax.plot(sub[param_name], sub[m], "o-", label=est, markersize=4)
    if r is not None:
        ax.axhline(r, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(t); ax.set_xlabel(param_name); ax.legend(fontsize=7)
fig.suptitle(f"Sensitivity to {param_name}", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### 3.2 Sensitivity to autocorrelation (β_pre)

In [ ]:
sweep2 = []
for val in [0.0, 0.2, 0.5, 0.8, 1.2, 1.5]:
    cfg_i = SimulationConfig(**{**cfg.__dict__, "beta_pre": val})
    _, summ_i = run_simulations(cfg_i, n_sim=n_sim_sweep)
    rho = cfg_i.effective_autocorrelation()
    for est_name, row in summ_i.iterrows():
        sweep2.append({"beta_pre": val, "rho": rho, "estimator": est_name, **row.to_dict()})

sweep2_df = pd.DataFrame(sweep2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, m, t in zip(axes, ["rmse", "coverage", "power"], ["RMSE", "Coverage", "Power"]):
    for est in focus:
        sub = sweep2_df[sweep2_df["estimator"] == est]
        if not sub.empty:
            ax.plot(sub["rho"], sub[m], "o-", label=est, markersize=4)
    if m == "coverage":
        ax.axhline(0.95, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(t); ax.set_xlabel("ρ(Y_pre, Y)"); ax.legend(fontsize=7)
fig.suptitle("Sensitivity to autocorrelation ρ", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### 3.3 Sensitivity to treatment-effect heterogeneity

In [ ]:
sweep3 = []
for val in [0.0, 0.02, 0.05, 0.10, 0.20, 0.50]:
    cfg_i = SimulationConfig(**{**cfg.__dict__, "sigma_heterogeneity": val})
    _, summ_i = run_simulations(cfg_i, n_sim=n_sim_sweep)
    for est_name, row in summ_i.iterrows():
        sweep3.append({"sigma_h": val, "estimator": est_name, **row.to_dict()})

sweep3_df = pd.DataFrame(sweep3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, m, t in zip(axes, ["bias", "rmse", "coverage"], ["Bias", "RMSE", "Coverage"]):
    for est in focus:
        sub = sweep3_df[sweep3_df["estimator"] == est]
        if not sub.empty:
            ax.plot(sub["sigma_h"], sub[m], "o-", label=est, markersize=4)
    if m == "bias":
        ax.axhline(0.0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    if m == "coverage":
        ax.axhline(0.95, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(t); ax.set_xlabel("σ_heterogeneity"); ax.legend(fontsize=7)
fig.suptitle("Sensitivity to treatment-effect heterogeneity", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

---
## Your own experiment

Change the config, run a single-experiment report **and** Monte Carlo.

In [ ]:
my_cfg = SimulationConfig(
    n_users       = 5_000,
    tau_true      = 0.10,
    beta_pre      = 1.2,
    sigma_post    = 0.8,
    p_new_users   = 0.30,
    sigma_heterogeneity = 0.05,
    prior_sd_tau  = 0.20,
)

# Single-experiment report
my_rng = np.random.default_rng(99)
my_data = simulate_user_level_data(my_cfg, my_rng)
my_report = experiment_report(my_data, my_cfg)
display(my_report.style.format({
    "τ̂": "{:.6f}", "SE": "{:.6f}",
    "95% CI lower": "{:.6f}", "95% CI upper": "{:.6f}",
    "t-stat": "{:.3f}", "p-value": "{:.4f}", "n": "{:.0f}",
}))

In [ ]:
# Monte Carlo for comparison
my_df, my_summary = run_simulations(my_cfg, n_sim=500)
display(my_summary.style.format("{:.5f}"))
fig = plot_summary_bars(my_summary)
plt.show()